In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import torch
from src.utils.utils import dotdict
from src.data_provider.data_loader import Dataset_Benchmark
from src.models.gaussian_diffusion import GaussianDiffusion
from scipy.stats import skew, kurtosis
from statsmodels.graphics.tsaplots import plot_acf

working_dir = r'F:\Projects\Ask2.ai\Diffusion'
os.chdir(working_dir)

sns.set_style('whitegrid')

In [ ]:
folder_path = './plots/information corruption/'

args = {'total_steps': 1000,
        'scheduler': 'cosine',
        'scale': 'z',}
args = dotdict(args)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
diffuser = GaussianDiffusion(args, device)

dataset = Dataset_Benchmark(data_path='./warehouse/processed/benchmark_data_log_ret.csv',
                            seq_len=128,
                            scale=args.scale,)
benchmark = dataset.data
benchmark = torch.tensor(benchmark).to(device)
df_benchmark = pd.read_csv('./warehouse/processed/benchmark_data_log_ret.csv')
labels = df_benchmark.columns[1:].to_list()

diffusion_steps = np.arange(0, 1000, 50)
benchmark = benchmark.unsqueeze(0).repeat(len(diffusion_steps), 1, 1)
t = torch.tensor(diffusion_steps).int().to(device)
x_noise, _ = diffuser.add_gauss_noise(benchmark, t=t)
x_noise = x_noise.detach().cpu().numpy()
noised_benchmark = {step: x_noise[i] for i, step in enumerate(diffusion_steps)}
noised_benchmark_inverse = {step: dataset.inverse_transform(x_noise[i]) for i, step in enumerate(diffusion_steps)}

# Moments

In [ ]:
mean_list = [np.mean(noised_benchmark[step], axis=0) for step in diffusion_steps]
std_list = [np.std(noised_benchmark[step], axis=0) for step in diffusion_steps]
skew_list = [skew(noised_benchmark[step], axis=0) for step in diffusion_steps]
kurt_list = [kurtosis(noised_benchmark[step], axis=0)+3 for step in diffusion_steps]

mean_list_inverse = [np.mean(noised_benchmark_inverse[step], axis=0) for step in diffusion_steps]
std_list_inverse = [np.std(noised_benchmark_inverse[step], axis=0) for step in diffusion_steps]
skew_list_inverse = [skew(noised_benchmark_inverse[step], axis=0) for step in diffusion_steps]
kurt_list_inverse = [kurtosis(noised_benchmark_inverse[step]+3, axis=0) for step in diffusion_steps]

In [ ]:
plt.figure(figsize=(15, 10))

plt.subplot(2, 2, 1)
plt.plot(diffusion_steps, mean_list)
plt.title('Mean')
plt.xlabel('Steps')
plt.ylabel('Mean')

plt.subplot(2, 2, 2)
plt.plot(diffusion_steps, std_list)
plt.title('Standard Deviation')
plt.xlabel('Steps')
plt.ylabel('Standard Deviation')

plt.subplot(2, 2, 3)
plt.plot(diffusion_steps, skew_list)
plt.title('Skewness')
plt.xlabel('Steps')
plt.ylabel('Skewness')

plt.subplot(2, 2, 4)
plt.plot(diffusion_steps, kurt_list)
plt.title('Kurtosis')
plt.xlabel('Steps')
plt.ylabel('Kurtosis')

plt.tight_layout()
plt.savefig(folder_path + f'moments_{args.scheduler}_{args.scale}_{args.total_steps}steps.png', dpi=300, bbox_inches='tight', pad_inches=0.2)
plt.show()

In [ ]:
plt.figure(figsize=(15, 10))

plt.subplot(2, 2, 1)
plt.plot(diffusion_steps, mean_list_inverse)
plt.title('Mean')
plt.xlabel('Steps')
plt.ylabel('Mean')

plt.subplot(2, 2, 2)
plt.plot(diffusion_steps, std_list_inverse)
plt.title('Standard Deviation')
plt.xlabel('Steps')
plt.ylabel('Standard Deviation')

plt.subplot(2, 2, 3)
plt.plot(diffusion_steps, skew_list_inverse)
plt.title('Skewness')
plt.xlabel('Steps')
plt.ylabel('Skewness')

plt.subplot(2, 2, 4)
plt.plot(diffusion_steps, kurt_list_inverse)
plt.title('Kurtosis')
plt.xlabel('Steps')
plt.ylabel('Kurtosis')

plt.tight_layout()
plt.savefig(folder_path + f'moments_{args.scheduler}_{args.scale}_{args.total_steps}steps_inverse.png', dpi=300, bbox_inches='tight', pad_inches=0.2)
plt.show()

# Covariance

In [ ]:
cov_list = [np.cov(noised_benchmark_inverse[step].T) for step in diffusion_steps]

In [ ]:
fig, axes = plt.subplots(5, 4, figsize=(15, 15))
axes = axes.flatten()

for i, step in enumerate(diffusion_steps):
    sns.heatmap(cov_list[i], ax=axes[i], cmap='coolwarm', center=0)
    axes[i].invert_yaxis()
    axes[i].set_title(f"Step {step}")
    
plt.suptitle('Covariance Matrix', y=1.005, fontsize=20)
plt.tight_layout()
plt.subplots_adjust(top=0.92) 

plt.savefig(folder_path + f'covariance_{args.scheduler}_{args.scale}_{args.total_steps}steps_inverse.png', dpi=300, bbox_inches='tight', pad_inches=0.2)
plt.show()

# Correlation

In [ ]:
corr_list = [np.corrcoef(noised_benchmark_inverse[step].T) for step in diffusion_steps]

In [ ]:
fig, axes = plt.subplots(5, 4, figsize=(15, 15))
axes = axes.flatten()

for i, step in enumerate(diffusion_steps):
    sns.heatmap(corr_list[i], ax=axes[i], cmap='coolwarm', center=0, vmax=1, vmin=-1)
    axes[i].invert_yaxis()
    axes[i].set_title(f"Step {step}")

plt.suptitle('Correlation Matrix', y=1.005, fontsize=20)
plt.tight_layout()
plt.subplots_adjust(top=0.92)
plt.savefig(folder_path + f'correlation_{args.scheduler}_{args.scale}_{args.total_steps}steps_inverse.png', dpi=300, bbox_inches='tight', pad_inches=0.2)
plt.show()

# Autocorrelation

In [ ]:
for i, label in enumerate(labels):
    ncolumns = 4
    nrows = 5
    fig, axes = plt.subplots(nrows, ncolumns, figsize=(15, 15))
    axes = axes.flatten()
    for j, step in enumerate(diffusion_steps):
        plot_acf(abs(noised_benchmark_inverse[step][:, i]), ax=axes[j], lags=100)
        axes[j].set_title(f"Step {step}")
    plt.suptitle(f'ACF of {label}', y=1.005, fontsize=20)
    plt.tight_layout()
    plt.subplots_adjust(top=0.92)
    plt.savefig(folder_path + f'ACF_{label}_{args.scheduler}_{args.scale}_{args.total_steps}steps_inverse.png', dpi=300, bbox_inches='tight', pad_inches=0.2)
    plt.clf()